# 24 — Demo bảo vệ: PhoBERT phẳng cho trang demo (Kaggle, GPU)

**Phạm vi**

Huấn luyện lại PhoBERT phẳng bốn lớp bằng cấu hình đã đóng băng (`03_configs/frozen/paper_neural_config.yaml`, mục `phobert.flat`),
seed 42, chỉ trên Train 1.200, rồi xuất trọng số cho trang demo `demo/app.py`. Notebook không đọc Validation, Hold-out hay Lazada và
không tạo kết quả đánh giá nào; mọi số liệu trong khóa luận giữ nguyên.

**Dữ liệu trên Kaggle**

Dataset đã dùng cho notebook 16, chỉ cần file `shopee_train.csv`.

**Yêu cầu trước khi chạy**

Bật GPU và Internet, có secret `GITHUB_TOKEN`. Mã huấn luyện (`02_src/src/classification/strategy_phobert/`) và cấu hình đã khóa đã có
trên GitHub từ bước 2c, không cần push thêm.

**Đầu ra**

File `/kaggle/working/phobert_flat.zip` gồm trọng số, tokenizer và `demo_meta.json`.

In [ ]:
import os, subprocess, sys, json, datetime, time, shutil
from kaggle_secrets import UserSecretsClient
REPO_URL = 'github.com/dnhphuong-05/kltn-vn-app-review-requirements.git'
REPO_DIR = '/kaggle/working/repo'
token = UserSecretsClient().get_secret('GITHUB_TOKEN')
if not os.path.exists(REPO_DIR):
    r = subprocess.run(['git', 'clone', '-q', f'https://{token}@{REPO_URL}', REPO_DIR], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('git clone thất bại: ' + r.stderr.replace(token, '***'))
subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin', f'https://{REPO_URL}'], check=True)
del token
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'py_vncorenlp', 'underthesea'], capture_output=True, text=True)
print('pip install py_vncorenlp underthesea:', 'ok' if r.returncode == 0 else r.stderr[-800:])
sys.path.append(os.path.join(REPO_DIR, '02_src'))
import numpy as np, pandas as pd, torch, transformers, yaml
from src.utils.seeds import set_seed
set_seed(42)
COMMIT = subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
print('Ngày chạy:', datetime.date.today().isoformat(), '| commit:', COMMIT)
print('transformers', transformers.__version__, '| torch', torch.__version__, '| GPU:', GPU)
assert GPU != 'cpu', 'Chưa bật GPU trong Settings → Accelerator.'

## Cấu hình và dữ liệu

**Cấu hình**

Mục `phobert.flat.config` của file đã đóng băng (amendment v3.4): 5 epoch, learning rate 3e-5, batch 16, max_len 256, warmup 10%,
weight decay 0,01, trọng số lớp cân bằng.

**Thứ tự lớp**

BUG, FEATURE, UX, OTHER, trùng với `src/demo/models.py` để trang demo đọc đúng xác suất của từng lớp.

In [ ]:
FZ = yaml.safe_load(open(os.path.join(REPO_DIR, '03_configs', 'frozen', 'paper_neural_config.yaml'), encoding='utf-8'))
CFG = FZ['paper_neural']['phobert']['flat']['config']
CLASSES = ['BUG', 'FEATURE', 'UX', 'OTHER']
SEED = 42
print('Cấu hình:', CFG)

def find_file(name, root='/kaggle/input'):
    hits = [os.path.join(d, name) for d, _, fs in os.walk(root) if name in fs]
    if not hits:
        raise FileNotFoundError(f'Không tìm thấy {name} trong {root}')
    return hits[0]

train_df = pd.read_csv(find_file('shopee_train.csv'), encoding='utf-8-sig')
assert len(train_df) == 1200 and set(train_df['gold_label']) == set(CLASSES)
y = train_df['gold_label'].map(CLASSES.index).tolist()
print('Train:', len(train_df), '| phân bố:', train_df['gold_label'].value_counts().to_dict())

## Tách từ và huấn luyện

**Bộ tách từ**

RDRSegmenter của VnCoreNLP như lần chạy trong khóa luận; nếu không khởi động được, notebook dừng thay vì đổi sang bộ tách từ khác,
để mô hình khớp với bộ tách từ mà trang demo dùng.

**Huấn luyện**

Hàm `train_phobert` của chiến lược PhoBERT, cùng vòng huấn luyện với lần chạy trong khóa luận.

In [ ]:
from src.classification.strategy_phobert.segment import Segmenter
from src.classification.strategy_phobert.train import train_phobert, load_tokenizer
SEG = Segmenter('vncorenlp', save_dir='/kaggle/working/vncorenlp')
seg_train = SEG(train_df['text'].astype(str).tolist())
print('Bộ tách từ:', SEG.backend, '| review tách lỗi:', SEG.failures)
print('Ví dụ:', SEG(['thủ tục hủy hàng rắc rối', 'app lag quá, thanh toán không được']))
model, hist = train_phobert(seg_train, y, len(CLASSES), CLASSES, CFG, SEED)
print(f"Huấn luyện {hist['train_seconds']:.1f} giây | loss theo epoch:", [round(x, 4) for x in hist['epoch_loss']])

## Lưu và nén

**Nội dung**

Thư mục `phobert_flat/` gồm trọng số (`model.safetensors`, `config.json`), tokenizer và `demo_meta.json` ghi cấu hình, commit, thời
gian huấn luyện và loss; nén thành `phobert_flat.zip` để tải về.

In [ ]:
OUT = '/kaggle/working/phobert_flat'
shutil.rmtree(OUT, ignore_errors=True)
model.save_pretrained(OUT)
load_tokenizer(CFG['model_name']).save_pretrained(OUT)
meta = {'created': datetime.datetime.now().isoformat(timespec='seconds'), 'notebook': '24_demo_phobert_kaggle', 'commit': COMMIT,
        'gpu': GPU, 'transformers': transformers.__version__, 'torch': torch.__version__, 'seed': SEED, 'classes': CLASSES,
        'config': CFG, 'segmenter': SEG.backend, 'train_seconds': hist['train_seconds'], 'epoch_loss': hist['epoch_loss'],
        'data': 'shopee_train.csv (Train 1.200); không dùng Validation, Hold-out, Lazada'}
json.dump(meta, open(os.path.join(OUT, 'demo_meta.json'), 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
shutil.make_archive('/kaggle/working/phobert_flat', 'zip', OUT)
print('Đã lưu:', sorted(os.listdir(OUT)))
print('File nén:', round(os.path.getsize('/kaggle/working/phobert_flat.zip') / 1e6, 1), 'MB')

## Mang về máy

**Các bước**

Tải `phobert_flat.zip` ở tab Output, giải nén vào `demo/models/phobert_flat/` (trong thư mục đó phải thấy ngay `config.json`), rồi chạy
`streamlit run demo/app.py`. Thư mục `demo/models/` không được commit.